# Key-Value Cache in Transformers

This notebook explains the **key-value cache**, often called the **KV cache**, using the Transformer implementation in [Attention_Is_All_You_Need.ipynb](Attention_Is_All_You_Need.ipynb) as the motivating example.

By the end, you should be able to explain:

- why autoregressive decoding repeats work,
- what keys and values are cached,
- why queries are still computed for the newest token,
- how cached decoding changes tensor shapes,
- how much memory the cache needs,
- why KV cache is mainly an inference optimization, not a new training objective.


## 1. The motivating example from the course notebook

In `Attention_Is_All_You_Need.ipynb`, training uses the full target sequence with a causal mask. That is efficient because all target positions can be processed in parallel.

Inference is different. The notebook's `greedy_decode` function generates one token at a time:

```python
target_ids = [SOS_IDX]

for i in range(max_len):
    target = torch.tensor(target_ids, dtype=torch.long).unsqueeze(0).to(device)
    target_mask = model.get_pad_mask(target, PAD_IDX) & model.get_subsequent_mask(target)

    with torch.no_grad():
        dec_output, attn = model.decoder(target, enc_output, source_mask, target_mask)
        output = model.generator(dec_output)

    target_id = output.argmax(dim=-1)[:, -1].item()
    target_ids.append(target_id)
```

Notice the important line:

```python
dec_output, attn = model.decoder(target, enc_output, source_mask, target_mask)
```

At every generation step, the decoder receives the **entire target prefix generated so far**.


## 2. What work is repeated?

Suppose the decoder is generating a German sentence and has produced these tokens:

```text
<s> ein mann laeuft
```

Without a KV cache, decoding looks like this:

| Step | Target prefix passed to decoder | Repeated work |
| --- | --- | --- |
| 1 | `<s>` | none |
| 2 | `<s> ein` | recompute attention projections for `<s>` |
| 3 | `<s> ein mann` | recompute attention projections for `<s> ein` |
| 4 | `<s> ein mann laeuft` | recompute attention projections for `<s> ein mann` |

The model only needs the newest output distribution, the one at the last position, but the current implementation recomputes decoder self-attention for the whole prefix each time.


## 3. Reminder: what are Q, K, and V?

In the course notebook, multi-head attention creates three learned projections:

```python
Q = self.W_q(query)
K = self.W_k(key)
V = self.W_v(value)
```

A useful interpretation is:

| Vector | Role in attention |
| --- | --- |
| Query `Q` | what the current position is looking for |
| Key `K` | what each available position offers as a match |
| Value `V` | what information is copied into the output if selected |

In decoder self-attention, `query`, `key`, and `value` all come from the target prefix:

```python
x1, _ = self.attn_layer(x, x, x, tgt_mask)
```

That means each generated token produces its own query, key, and value vectors in every decoder layer.


## 4. The key idea of KV cache

During autoregressive inference, previous tokens do not change.

Once the model has computed the key and value vectors for old tokens, it can store them and reuse them at later steps.

At a new decoding step:

1. Embed only the newest token.
2. Compute its new query, key, and value.
3. Append the new key and value to the cached keys and values from earlier steps.
4. Let the newest query attend over all cached keys and values.
5. Produce logits for the next token.

So the cache stores **K and V**, not Q:

| Tensor | Cache it? | Why? |
| --- | --- | --- |
| `Q` | No | The query belongs to the newest position asking for information. |
| `K` | Yes | Old keys are still valid match targets for future tokens. |
| `V` | Yes | Old values are still valid information sources for future tokens. |


## 5. Shape comparison

Using the notebook's notation:

- `batch_size = B`
- `target_seq_len = T`
- `d_model = 512`
- `n_heads = 8`
- `d_k = d_model // n_heads = 64`

Without cache, decoder self-attention at step `T` uses:

```text
Q: (B, n_heads, T, d_k)
K: (B, n_heads, T, d_k)
V: (B, n_heads, T, d_k)
```

With cache, the newest token has length 1:

```text
new_Q:       (B, n_heads, 1, d_k)
new_K:       (B, n_heads, 1, d_k)
new_V:       (B, n_heads, 1, d_k)
cached_K:    (B, n_heads, T - 1, d_k)
cached_V:    (B, n_heads, T - 1, d_k)
combined_K:  (B, n_heads, T, d_k)
combined_V:  (B, n_heads, T, d_k)
```

The attention scores for the newest token become:

```text
scores = new_Q @ combined_K.transpose(-2, -1)
scores shape = (B, n_heads, 1, T)
```

The newest token can attend to all previous tokens, but we did not recompute old keys and values.


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

B = 1
T = 5
d_model = 8
n_heads = 2
d_k = d_model // n_heads

print(f"B={B}, T={T}, d_model={d_model}, n_heads={n_heads}, d_k={d_k}")


B=1, T=5, d_model=8, n_heads=2, d_k=4


## 6. A small attention module for the cache demonstration

The class below mirrors the course notebook's attention structure:

```python
self.W_q = nn.Linear(d_model, d_model, bias=False)
self.W_k = nn.Linear(d_model, d_model, bias=False)
self.W_v = nn.Linear(d_model, d_model, bias=False)
self.W_o = nn.Linear(d_model, d_model)
```

For the demonstration, we remove dropout so that full-prefix decoding and cached decoding can be compared exactly.


In [2]:
class TinyCachedSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size = x.size(0)
        return x.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

    def group_heads(self, x):
        batch_size = x.size(0)
        return x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

    def attend(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        probs = F.softmax(scores, dim=-1)
        output = torch.matmul(probs, V)
        return output, probs

    def forward_full_prefix(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        seq_len = x.size(1)
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

        output, probs = self.attend(Q, K, V, causal_mask)
        output = self.W_o(self.group_heads(output))
        return output, probs

    def forward_one_step_with_cache(self, x_new, cache=None):
        # This method represents one autoregressive decoding step.
        # Instead of receiving the whole prefix, it receives only the newest
        # token representation.
        #
        # x_new: (B, 1, d_model)
        #   B is the batch size.
        #   1 is the sequence length for this step: only the newest token.
        #   d_model is the model width before splitting into attention heads.

        # The newest token still needs a query because it is the position that
        # is asking, "Which previous tokens should I attend to?"
        # Shape after split_heads: (B, n_heads, 1, d_k)
        new_Q = self.split_heads(self.W_q(x_new))

        # The newest token also contributes a key and a value. These will be
        # useful not only now, but also for future generated tokens, so they
        # are the tensors we add to the cache.
        # Shape after split_heads: (B, n_heads, 1, d_k)
        new_K = self.split_heads(self.W_k(x_new))
        new_V = self.split_heads(self.W_v(x_new))

        if cache is None:
            # First decoding step: there are no previous target tokens.
            # The complete key/value memory is just the newest token's K and V.
            # combined_K, combined_V: (B, n_heads, 1, d_k)
            combined_K = new_K
            combined_V = new_V
        else:
            # Later decoding steps: cache["K"] and cache["V"] contain the keys
            # and values for all earlier generated tokens in this decoder layer.
            #
            # cache["K"]: (B, n_heads, previous_len, d_k)
            # new_K:      (B, n_heads, 1,            d_k)
            #
            # Concatenate along dim=2, the sequence-length dimension, so the
            # newest token is appended after the earlier tokens.
            #
            # combined_K: (B, n_heads, previous_len + 1, d_k)
            # combined_V: (B, n_heads, previous_len + 1, d_k)
            combined_K = torch.cat([cache["K"], new_K], dim=2)
            combined_V = torch.cat([cache["V"], new_V], dim=2)

        # Attention for this step uses only the newest query, but it can look
        # across all keys and values generated so far.
        #
        # new_Q:      (B, n_heads, 1,               d_k)
        # combined_K: (B, n_heads, current_seq_len, d_k)
        # combined_V: (B, n_heads, current_seq_len, d_k)
        #
        # scores inside attend(): (B, n_heads, 1, current_seq_len)
        # output from attend():   (B, n_heads, 1, d_k)
        #
        # No causal mask is needed in this one-step version because the cache
        # contains only past tokens plus the current token. Future tokens have
        # not been generated yet, so there is nothing future to hide.
        output, probs = self.attend(new_Q, combined_K, combined_V)

        # Merge the attention heads back to model width and apply the output
        # projection, matching the structure in the course notebook's
        # MultiHeadAttention.forward method.
        # Before group_heads: (B, n_heads, 1, d_k)
        # After group_heads:  (B, 1, d_model)
        output = self.W_o(self.group_heads(output))

        # Return the updated cache so the next decoding step can reuse these
        # K and V tensors instead of recomputing them from the old tokens.
        # Notice that Q is not cached: the next step will compute a fresh query
        # for the next newest token.
        new_cache = {"K": combined_K, "V": combined_V}
        return output, probs, new_cache


## 7. Full-prefix decoding versus cached decoding

We now compare two ways to compute the representation of the newest token:

1. **Full-prefix method:** pass all `T` tokens and take the final position.
2. **Cached method:** feed one token at a time, store K and V, and take the output from the final step.

For causal self-attention, these should match for the final token when dropout is disabled and the same weights are used.


In [3]:
# Create one attention module. Both decoding methods below use this same
# module instance, so they share exactly the same learned weights.
attn = TinyCachedSelfAttention(d_model=d_model, n_heads=n_heads)

# Turn off training-time behavior. This small module has no dropout, but using
# eval() reinforces the inference setting where KV cache is normally used.
attn.eval()

# Make one fake target-prefix representation with T tokens.
# x: (B, T, d_model)
# Think of this as the embedded/hidden representation of a generated prefix,
# such as: <s> ein mann laeuft ...
x = torch.randn(B, T, d_model)

# We are comparing two inference strategies, not training. Gradients are not
# needed, so torch.no_grad() saves memory and matches normal generation usage.
with torch.no_grad():
    # Strategy 1: full-prefix decoding.
    # Pass the whole prefix of length T through causal self-attention in one go.
    # full_output: (B, T, d_model)
    full_output, full_probs = attn.forward_full_prefix(x)

    # During generation we only use the newest position to predict the next
    # token. Therefore, the final position is the relevant output to compare.
    # full_last_output: (B, 1, d_model)
    full_last_output = full_output[:, -1:, :]

    # Strategy 2: cached decoding.
    # Start with an empty cache because no tokens have been processed yet.
    cache = None

    # Keep each one-step output so we can inspect the final step later.
    cached_outputs = []

    # Feed the same prefix one token at a time, exactly like autoregressive
    # generation. At step t, the model sees only x[:, t:t+1, :] as the newest
    # token, while old keys and values arrive through the cache.
    for t in range(T):
        # Select one token while preserving the sequence dimension.
        # x_new: (B, 1, d_model), not (B, d_model)
        x_new = x[:, t:t+1, :]

        # step_output: representation for this newest token, (B, 1, d_model)
        # step_probs: attention weights for this newest token over all cached
        #             positions so far, (B, n_heads, 1, t + 1)
        # cache: updated dictionary containing K and V for positions 0..t
        step_output, step_probs, cache = attn.forward_one_step_with_cache(x_new, cache)

        cached_outputs.append(step_output)

    # The final cached output should represent the same final token as
    # full_last_output above.
    # cached_last_output: (B, 1, d_model)
    cached_last_output = cached_outputs[-1]

# Print the main shape checks.
print("Full-prefix final-token output shape:", tuple(full_last_output.shape))
print("Cached final-token output shape:     ", tuple(cached_last_output.shape))

# After T one-token steps, the cache should contain keys and values for all T
# positions. The sequence-length dimension is dim=2:
# cache["K"], cache["V"]: (B, n_heads, T, d_k)
print("Cached K shape after all steps:      ", tuple(cache["K"].shape))
print("Cached V shape after all steps:      ", tuple(cache["V"].shape))

# If the cache is implemented correctly, the final-token output from cached
# decoding should match the final-token output from full-prefix causal decoding.
# Small floating-point differences are allowed through atol=1e-6.
print("Outputs match:", torch.allclose(full_last_output, cached_last_output, atol=1e-6))


Full-prefix final-token output shape: (1, 1, 8)
Cached final-token output shape:      (1, 1, 8)
Cached K shape after all steps:       (1, 2, 5, 4)
Cached V shape after all steps:       (1, 2, 5, 4)
Outputs match: True


## 8. Why this matters computationally

In the course notebook's `greedy_decode`, step `T` sends a prefix of length `T` through every decoder layer. The self-attention projections for old positions are recomputed at each step.

For a long generated sequence, that repeated work grows quickly:

```text
without cache: process prefixes of length 1 + 2 + 3 + ... + T
with cache:    process one new token at each step, while attending to cached K,V
```

The model still attends over all previous positions, so attention over the context is not free. But the expensive repeated projection and layer computation for old tokens is avoided in decoder self-attention.

This is why KV cache is central to fast text generation in GPT-style models and decoder-side generation in encoder-decoder models.


## 9. How this maps to the notebook's encoder-decoder Transformer

The course implementation has two decoder attention blocks:

```python
# Decoder self-attention
x1, _ = self.attn_layer(x, x, x, tgt_mask)

# Encoder-decoder attention
x1, attn = self.enc_attn_layer(x, memory, memory, src_mask)
```

KV cache applies most directly to the first block, **decoder self-attention**. Previous target tokens produce reusable keys and values.

The second block, **encoder-decoder attention**, already has a related reuse pattern:

- `memory` is the encoder output.
- It does not change during decoding.
- It is used as both key and value in every decoder layer.

A production implementation often precomputes the cross-attention K and V from `memory` once per decoder layer, then reuses them for every generated token. That is conceptually similar to caching, but the common phrase "KV cache" usually refers to caching previous decoder self-attention keys and values during autoregressive generation.


## 10. What would need to change in the course implementation?

The current notebook implementation is intentionally simple and clear:

```python
def forward(self, query, key, value, mask=None):
    Q = self.W_q(query)
    K = self.W_k(key)
    V = self.W_v(value)
    ...
```

To implement KV cache, the model would need a generation-specific path:

```python
def forward_step(self, x_new, memory, src_mask, layer_cache):
    # 1. compute Q,K,V only for x_new in decoder self-attention
    # 2. append new K,V to layer_cache
    # 3. attend with Q over cached K,V
    # 4. reuse encoder memory K,V for cross-attention
    # 5. return newest output and updated cache
```

The cache must be kept separately for each decoder layer because every layer has different learned projection matrices and different hidden states.

For a 6-layer decoder, the cache is not one tensor. It is usually a list or dictionary containing one K tensor and one V tensor per decoder layer.


## 11. Pseudocode: cached greedy decoding

This is the same idea as `greedy_decode`, but instead of repeatedly passing the whole `target` prefix, the decoder receives only the newest token and an updated cache.

```python
def greedy_decode_with_cache(model, sentence, max_len=100):
    source = make_source_tensor(sentence)
    source_mask = model.get_pad_mask(source, PAD_IDX)

    enc_output = model.encoder(source, source_mask)

    target_id = SOS_IDX
    target_ids = [target_id]
    cache = initialize_empty_cache(num_decoder_layers=model.decoder.n_layers)

    for step in range(max_len):
        newest_token = torch.tensor([[target_id]], device=device)

        dec_output, cache, attn = model.decoder.forward_step(
            newest_token,
            enc_output,
            source_mask,
            cache,
            step=step,
        )

        logits = model.generator(dec_output[:, -1:, :])
        target_id = logits.argmax(dim=-1).item()
        target_ids.append(target_id)

        if target_id == EOS_IDX:
            break

    return target_ids
```

The notebook's current implementation does not define `forward_step`; adding it would be an advanced extension.


## 12. How much memory does the cache use?

The cache trades computation for memory: every key and value we keep must be stored until generation ends. We can count exactly how many numbers that is.

| Symbol | Meaning |
| --- | --- |
| `B` | batch size (number of sequences generated together) |
| `T` | number of cached tokens: the prompt plus the tokens generated so far |
| `L` | number of decoder layers |
| `n_heads` | number of attention heads |
| `d_k` | size of each head's key and value vectors |
| `d_model` | model width, equal to `n_heads * d_k` |
| `bytes_per_number` | 4 for fp32; 2 for fp16 or bf16 |

From section 5, one decoder layer caches one K tensor and one V tensor, each of shape `(B, n_heads, T, d_k)`. Counting the numbers:

```text
K, one layer:             B * n_heads * T * d_k
K and V, one layer:   2 * B * n_heads * T * d_k
K and V, all layers:  2 * B * L * n_heads * T * d_k
```

Multiplying by the size of one number gives the memory:

```text
KV cache bytes = 2 * B * L * n_heads * d_k * T * bytes_per_number
               = (2 * L * d_model * bytes_per_number) * B * T
```

The second line uses `d_model = n_heads * d_k`. The bracketed term is the **cost of one token in one sequence**. It is fixed by the architecture and the number format, so the cache grows linearly with the batch size `B` and with the number of tokens `T`. Because `T` increases by one at every generation step, the cache keeps growing while the model generates.

What the formula assumes:

- **Every head has its own keys and values**, as in standard multi-head attention (this notebook and GPT-2). Multi-query and grouped-query attention share K and V across heads; replace `n_heads` with the number of K/V heads, `n_kv_heads`. The shortcut `2 * L * d_model` then overestimates the cache. See [attention_mechanisms.ipynb](attention_mechanisms.ipynb).
- **Only decoder self-attention is counted.** If an encoder-decoder model also precomputes the cross-attention K and V from the encoder output (section 9), add `2 * B * L * S * d_model * bytes_per_number`, where `S` is the source length. That part is fixed for a given source and does not grow during decoding.
- **Only the cache is counted**, not the model weights, other activations, or memory reserved by the framework.

### Worked example: GPT-2 small

GPT-2 small is a decoder-only model with standard multi-head attention, so the shortcut applies directly. Its configuration ([`config.json` on Hugging Face](https://huggingface.co/openai-community/gpt2/blob/main/config.json)) gives `n_layer = 12`, `n_embd = 768` (our `d_model`), `n_head = 12` (so `d_k = 64`), and a maximum context of `n_positions = 1024` tokens.

The next cell applies the formula, then checks it by allocating an empty cache with the shapes from section 5 and adding up its bytes. It needs no download; the full-context check allocates about 38 MB on the CPU.

In [ ]:
# GPT-2 small configuration (openai-community/gpt2 config.json).
# Prefixed with gpt2_ so the toy values from cell 6 (B, T, d_model, ...) are not overwritten.
gpt2_L = 12                            # n_layer: decoder blocks
gpt2_d_model = 768                     # n_embd
gpt2_n_heads = 12                      # n_head
gpt2_d_k = gpt2_d_model // gpt2_n_heads  # 64
gpt2_max_T = 1024                      # n_positions: longest context GPT-2 small accepts


def kv_cache_bytes(B, T, L, n_heads, d_k, bytes_per_number):
    """K and V, for every layer, every head, every cached token."""
    return 2 * B * L * n_heads * d_k * T * bytes_per_number


for fmt, nbytes in [("fp32", 4), ("fp16/bf16", 2)]:
    per_token = 2 * gpt2_L * gpt2_d_model * nbytes   # the bracketed term
    print(f"{fmt}: {per_token:,} bytes per token per sequence")
    for batch, tokens in [(1, 1024), (8, 1024)]:
        total = kv_cache_bytes(batch, tokens, gpt2_L, gpt2_n_heads, gpt2_d_k, nbytes)
        assert total == per_token * batch * tokens   # both forms of the formula agree
        print(f"  B={batch}, T={tokens}: {total:,} bytes = {total / 1e6:.1f} MB")

# Check against real tensors: one (K, V) pair per layer, each (B, n_heads, T, d_k), in fp16.
batch, tokens = 1, gpt2_max_T
gpt2_cache = [
    (torch.empty(batch, gpt2_n_heads, tokens, gpt2_d_k, dtype=torch.float16),
     torch.empty(batch, gpt2_n_heads, tokens, gpt2_d_k, dtype=torch.float16))
    for _ in range(gpt2_L)
]
allocated = sum(t.numel() * t.element_size() for pair in gpt2_cache for t in pair)
print(f"\nAllocated fp16 cache for B={batch}, T={tokens}: {allocated:,} bytes")
print("Matches formula:", allocated == kv_cache_bytes(batch, tokens, gpt2_L, gpt2_n_heads, gpt2_d_k, 2))
del gpt2_cache

Expected output (from the formula; 1 MB = 10<sup>6</sup> bytes):

| Format | Per token, one sequence | `B = 1`, `T = 1024` | `B = 8`, `T = 1024` |
| --- | --- | --- | --- |
| fp32 | 73,728 bytes | 75.5 MB | 604.0 MB |
| fp16 / bf16 | 36,864 bytes | 37.7 MB | 302.0 MB |

How to read these numbers:

- One token costs `2 * 12 * 768 = 18,432` stored numbers, whatever the prompt says.
- GPT-2 small has about 124 million parameters, roughly 249 MB in fp16. A batch of eight full-length sequences already needs more memory for the cache (302 MB) than for the weights.
- Larger models multiply both `L` and `d_model`, and modern models also allow much longer contexts. That is why the KV cache, not only the weights, often limits batch size and context length when serving a model. Multi-query and grouped-query attention reduce it by storing fewer K/V heads.

## 13. Common misconceptions

| Misconception | Correction |
| --- | --- |
| KV cache changes the model's learned behavior. | It should not. It is an inference optimization that reuses already-computed tensors. |
| KV cache removes the need to attend to old tokens. | No. The newest query still attends over old keys and values. |
| KV cache is useful during normal teacher-forced training. | Usually no. Training uses full target sequences in parallel with a causal mask. |
| There is one global cache for the whole decoder. | Usually no. Each decoder layer needs its own cached K and V. |
| We cache Q, K, and V. | Usually only K and V are cached. The newest query is computed at the current step. |


## 14. Quick check questions

1. In the original `greedy_decode`, why does the decoder recompute work for old target tokens?
2. Why are keys and values reusable, but the newest query is still computed?
3. In decoder self-attention with cache, what is the shape of `new_Q` when generating one token?
4. Why does each decoder layer need its own cache?
5. Why is KV cache more relevant for inference than for teacher-forced training?
